# IFC nach RDF mit IFCtoLBD (LBD Level 1)

Dieses Notebook ist fuer die Ausfuehrung in **Renku** vorbereitet.

- IFC Quelle: `/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/Models/FoC_Preserving_IFC-Gantenbein.ifc`
- IFCtoLBD Fork: `/home/renku/work/IFCtoLBD`
- Ausgabeordner: `/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data`
- LBD Modell: **Level 1**

In [ ]:
# Optional: nur falls in der Umgebung noch nicht vorhanden
!pip install --quiet rdflib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.5/496.5 kB 17.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from datetime import datetime
from pathlib import Path
import shutil
import subprocess

# ===== Konfiguration (Renku Pfade) =====
ifc_input = Path("/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/Models/FoC_Preserving_IFC-Gantenbein.ifc")
ifctolbd_repo = Path("/home/renku/work/IFCtoLBD")
ifctolbd_cli_jar = ifctolbd_repo / "IFCtoLBD_CLI.jar"
output_dir = Path("/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data")

# Optional: URI-Basis fuer erzeugte Ressourcen
base_uri = "http://dca.example.org/"

# ===== Vorbedingungen mit klaren Fehlermeldungen =====
if shutil.which("java") is None:
    raise EnvironmentError(
        "Java wurde nicht gefunden. Bitte in Renku ein Java Runtime Environment (OpenJDK 17/21) bereitstellen."
    )

if not ifc_input.exists():
    raise FileNotFoundError(f"IFC-Datei nicht gefunden: {ifc_input}")

if not ifctolbd_repo.exists():
    raise FileNotFoundError(f"IFCtoLBD Repository nicht gefunden: {ifctolbd_repo}")

if not ifctolbd_cli_jar.exists():
    raise FileNotFoundError(
        f"IFCtoLBD_CLI.jar nicht gefunden: {ifctolbd_cli_jar}. "
        "Bitte JAR im Fork bereitstellen oder Pfad anpassen."
    )

output_dir.mkdir(parents=True, exist_ok=True)

date_stamp = datetime.now().strftime("%Y%m%d")
output_file = output_dir / f"{ifc_input.stem}_{date_stamp}.ttl"

# ===== IFC -> RDF (LBD Level 1) via CLI =====
cmd = [
    "java",
    "-jar",
    str(ifctolbd_cli_jar),
    str(ifc_input),
    "--level",
    "1",
    "--target_file",
    str(output_file),
    "--url",
    base_uri,
]

print("Starte Konvertierung:")
print(" ".join(cmd))

# Live-Ausgabe waehrend der Laufzeit
process = subprocess.Popen(
    cmd,
    cwd=ifctolbd_repo,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"IFCtoLBD Konvertierung fehlgeschlagen (Returncode {return_code}).")

if not output_file.exists():
    raise FileNotFoundError(
        f"Konvertierung beendet, aber Ausgabedatei fehlt: {output_file}"
    )

print("\nKonvertierung erfolgreich.")
print(f"Ausgabe gespeichert: {output_file}")

JVMNotFoundException: No JVM shared library file (libjvm.so) found. Try setting up the JAVA_HOME environment variable properly.